In [19]:
!pip install faiss-cpu
!pip install joblib
!pip install umap-learn


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [20]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

import faiss
import joblib


In [21]:
import os

# Try common locations for data.csv so the notebook doesn't fail with FileNotFoundError
candidates = [
	"./data.csv",
	"../data.csv",
	os.path.join(os.getcwd(), "data.csv"),
	os.path.expanduser("~/data.csv"),
	"/data.csv"
]

csv_path = next((p for p in candidates if os.path.exists(p)), None)

if csv_path is None:
	present_csvs = [f for f in os.listdir('.') if f.lower().endswith('.csv')]
	raise FileNotFoundError(
		"data.csv not found. Tried paths:\n  " + "\n  ".join(candidates) +
		"\nCSV files in current directory:\n  " + ("\n  ".join(present_csvs) if present_csvs else "None") +
		"\nPlease upload 'data.csv' to the notebook working directory or update the path."
	)

df = pd.read_csv(csv_path)
df.head()


,valence,year,acousticness,artists,danceability,duration_ms,energy,explicit,id,instrumentalness,key,liveness,loudness,mode,name,popularity,release_date,speechiness,tempo
0,0.0594,1921,0.982,"['Sergei Rachmaninoff', 'James Levine', 'Berli...",0.279,831667,0.211,0,4BJqT0PrAfrxzMOxytFOIz,0.878000,10,0.665,-20.096,1,"Piano Concerto No. 3 in D Minor, Op. 30: III. ...",4,1921,0.0366,80.954
1,0.9630,1921,0.732,['Dennis Day'],0.819,180533,0.341,0,7xPhfUan2yNtyFG0cUWkt8,0.000000,7,0.160,-12.441,1,Clancy Lowered the Boom,5,1921,0.4150,60.936
2,0.0394,1921,0.961,['KHP Kridhamardawa Karaton Ngayogyakarta Hadi...,0.328,500062,0.166,0,1o6I8BglA6ylDMrIELygv1,0.913000,3,0.101,-14.850,1,Gati Bali,5,1921,0.0339,110.339
3,0.1650,1921,0.967,['Frank Parker'],0.275,210000,0.309,0,3ftBPsC5vPBKxYSee08FDH,0.000028,5,0.381,-9.316,1,Danny Boy,3,1921,0.0354,100.109
4,0.2530,1921,0.957,['Phil Regan'],0.418,166693,0.193,0,4d6HGyGT8e121BsdKmw9v6,0.000002,3,0.229,-10.096,1,When Irish Eyes Are Smiling,2,1921,0.0380,101.665


In [22]:
df["genre"] = df["artists"].astype(str).apply(lambda x: x.split(",")[0])
df["genre"] = df["genre"].replace("", "unknown")
df["genre"].fillna("unknown", inplace=True)

df.genre.value_counts().head()


C:\Users\Mrigyank Roy\AppData\Local\Temp\ipykernel_5676\3336834458.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["genre"].fillna("unknown", inplace=True)


genre
['Francisco Canaro'      1285
['Эрнест Хемингуэй']     1211
['Эрих Мария Ремарк']    1068
['Frédéric Chopin'       1016
['Francisco Canaro']      942
Name: count, dtype: int64

In [23]:
feature_cols = [
    "danceability","energy","key","loudness","speechiness",
    "acousticness","instrumentalness","liveness","valence",
    "tempo","duration_ms"
]

X = df[feature_cols].astype("float32")


In [24]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

joblib.dump(scaler, "scaler.pkl")
np.save("features.npy", X_scaled)

X_scaled[:5]


array([[-1.4670126 , -1.0139884 ,  1.365588  , -1.514237  , -0.37970635,
         1.2761866 ,  2.268102  ,  2.626719  , -1.7828246 , -1.1693068 ,
         4.7631464 ],
       [ 1.5987788 , -0.52827024,  0.5121232 , -0.17076562,  1.9454806 ,
         0.61134714, -0.5327705 , -0.26222867,  1.6506883 , -1.8211796 ,
        -0.399747  ],
       [-1.1888205 , -1.1821216 , -0.6258298 , -0.5935511 , -0.3962973 ,
         1.2203401 ,  2.3797538 , -0.5997492 , -1.858821  , -0.21240388,
         2.1338239 ],
       [-1.4897223 , -0.6478317 , -0.05685331,  0.3776795 , -0.38708013,
         1.2362962 , -0.5326822 ,  1.0020435 , -1.3815641 , -0.5455369 ,
        -0.16610081],
       [-0.6778552 , -1.0812416 , -0.6258298 ,  0.24078766, -0.37110367,
         1.2097026 , -0.5327652 ,  0.13249883, -1.0471805 , -0.4948668 ,
        -0.5094855 ]], dtype=float32)

In [25]:
class TripletDataset(torch.utils.data.Dataset):
    def __init__(self, df, features):
        self.df = df.reset_index(drop=True)
        self.X = features

        # Encode genres
        self.df["genre_id"] = self.df["genre"].astype("category").cat.codes
        self.num_genres = self.df["genre_id"].nunique()

        # Group songs by genre
        self.genre_groups = {
            g: self.df[self.df.genre_id == g].index.tolist()
            for g in self.df["genre_id"].unique()
        }

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        anchor = idx
        anchor_genre = self.df.loc[idx, "genre_id"]

        # Positive sample
        pos = np.random.choice(self.genre_groups[anchor_genre])

        # Negative sample
        neg_genre = np.random.choice(
            [g for g in self.genre_groups.keys() if g != anchor_genre]
        )
        neg = np.random.choice(self.genre_groups[neg_genre])

        return (
            torch.tensor(self.X[anchor], dtype=torch.float32),
            torch.tensor(self.X[pos], dtype=torch.float32),
            torch.tensor(self.X[neg], dtype=torch.float32),
            torch.tensor(anchor_genre)
        )


In [26]:
class MusicEmbeddingNet(nn.Module):
    def __init__(self, num_genres):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(11, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 32)
        )

        self.genre_head = nn.Linear(32, num_genres)

    def forward(self, x):
        z = self.encoder(x)
        genre_pred = self.genre_head(z)
        return z, genre_pred


class TripletLoss(nn.Module):
    def __init__(self, margin=1.0):
        super().__init__()
        self.margin = margin

    def forward(self, anchor, positive, negative):
        pos_dist = F.pairwise_distance(anchor, positive)
        neg_dist = F.pairwise_distance(anchor, negative)
        return torch.relu(pos_dist - neg_dist + self.margin).mean()


In [27]:
dataset = TripletDataset(df, X_scaled)
loader = torch.utils.data.DataLoader(dataset, batch_size=128, shuffle=True)

num_genres = dataset.num_genres
model = MusicEmbeddingNet(num_genres)

triplet_loss = TripletLoss()
genre_loss_fn = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=1e-3)

EPOCHS = 30

for epoch in range(EPOCHS):
    epoch_loss = []

    for anchor, pos, neg, genre_id in loader:
        optimizer.zero_grad()

        # Forward
        z_a, g_pred_a = model(anchor)
        z_p, _ = model(pos)
        z_n, _ = model(neg)

        # Losses
        loss_triplet = triplet_loss(z_a, z_p, z_n)
        loss_genre = genre_loss_fn(g_pred_a, genre_id.long())

        loss = loss_triplet + 0.5 * loss_genre
        loss.backward()
        optimizer.step()

        epoch_loss.append(loss.item())

    print(f"Epoch {epoch+1}: Loss = {np.mean(epoch_loss):.4f}")
      
torch.save(model.state_dict(), "embedding_model.pt")


Epoch 1: Loss = 4.4310
Epoch 2: Loss = 3.8934
Epoch 3: Loss = 3.7069
Epoch 4: Loss = 3.5839
Epoch 5: Loss = 3.4854
Epoch 6: Loss = 3.4132
Epoch 7: Loss = 3.3426
Epoch 8: Loss = 3.2828
Epoch 9: Loss = 3.2270
Epoch 10: Loss = 3.1813
Epoch 11: Loss = 3.1306
Epoch 12: Loss = 3.0918
Epoch 13: Loss = 3.0488
Epoch 14: Loss = 3.0109
Epoch 15: Loss = 2.9808
Epoch 16: Loss = 2.9520
Epoch 17: Loss = 2.9243
Epoch 18: Loss = 2.8958
Epoch 19: Loss = 2.8679
Epoch 20: Loss = 2.8397
Epoch 21: Loss = 2.8259
Epoch 22: Loss = 2.8059
Epoch 23: Loss = 2.7889
Epoch 24: Loss = 2.7673
Epoch 25: Loss = 2.7571
Epoch 26: Loss = 2.7317
Epoch 27: Loss = 2.7201
Epoch 28: Loss = 2.7093
Epoch 29: Loss = 2.6869
Epoch 30: Loss = 2.6790


In [28]:
model.eval()
with torch.no_grad():
    embeddings = model.encoder(torch.tensor(X_scaled)).numpy()

np.save("embeddings.npy", embeddings)
embeddings[:5]


array([[-1.4489632 , -1.4623528 ,  2.2216463 , -3.9335377 , -3.1438963 ,
        -0.75328684, -1.7026719 , -4.605097  ,  4.335235  , -6.728626  ,
         0.7716187 ,  0.2633452 , -0.75207317, -0.5616174 ,  3.8337846 ,
         3.1380944 ,  3.278459  , -2.5464709 , -0.38442457,  4.6209426 ,
         0.9521935 ,  0.61662096, -1.2708163 ,  0.9491521 ,  3.1724014 ,
        -0.26289374, -1.1905262 , -0.32770202, -3.2350688 ,  3.1555324 ,
        -0.82820106, -2.4220877 ],
       [ 5.235512  ,  1.7269008 ,  0.55322635, -0.9037988 ,  1.7270378 ,
        -3.0138707 , -1.1414949 , -1.9933387 , -6.694838  , -2.6989176 ,
         4.6946135 ,  1.4642043 , -4.818947  , -3.5341485 , -4.9916463 ,
        -2.2757752 , -3.1860607 , -0.9866859 ,  3.860456  ,  1.5009093 ,
         0.41234776, -1.562542  , -0.01999164, -1.5803164 ,  3.0382137 ,
         0.79993135, -3.0068412 ,  1.1697248 ,  3.4965615 , -3.6921625 ,
        -1.1742866 ,  0.893829  ],
       [-1.5598757 ,  0.96910286,  1.3853993 , -1.3352

In [29]:
emb = embeddings.astype("float32")
emb = emb / np.linalg.norm(emb, axis=1, keepdims=True)

d = emb.shape[1]
nlist = 100
m = 8

quantizer = faiss.IndexFlatIP(d)
index = faiss.IndexIVFPQ(quantizer, d, nlist, m, 8)

index.train(emb)
index.add(emb)

faiss.write_index(index, "faiss.index")

print("FAISS index built!")


FAISS index built!


In [30]:
df_display = df[["name","artists","genre"]]

def recommend(idx, k=10):
    vec = emb[idx].reshape(1,-1)
    _, indices = index.search(vec, k+1)
    rec = df_display.iloc[indices[0][1:]]
    return rec

recommend(10)


,name,artists,genre
38821,Pebeta Canyengue - Remasterizado,['Ignacio Corsini'],['Ignacio Corsini']
39252,Rayito de Sol - Remasterizado,['Ignacio Corsini'],['Ignacio Corsini']
35,Overture,"['Ermanno Wolf-Ferrari', 'Arturo Toscanini']",['Ermanno Wolf-Ferrari'
38940,Bicho Feo - Remasterizado,['Ignacio Corsini'],['Ignacio Corsini']
75532,Nelly - Remasterizado,['Ignacio Corsini'],['Ignacio Corsini']
20946,Jamás Pobre Olvidarte - Remasterizado,['Ignacio Corsini'],['Ignacio Corsini']
111466,"Violin Sonata in D Major, RV 10: II. Allegro","['Antonio Vivaldi', 'Ottorino Respighi', 'Joha...",['Antonio Vivaldi'
146,Quand Il Y A Une Femme Dans Un Coin,['Maurice Chevalier'],['Maurice Chevalier']
109114,Organito - Remasterizado,['Ignacio Corsini'],['Ignacio Corsini']
390,Aime Moi Emma,['Dranem'],['Dranem']


In [37]:
def precision_at_k(i, k=30):
    genre = df.loc[i, "genre"]
    _, idx = index.search(emb[i].reshape(1,-1), k+1)
    rec_genres = df.iloc[idx[0][1:]]["genre"]
    return np.mean(rec_genres == genre)

def recall_at_k(i, k=30):
    genre = df.loc[i, "genre"]
    total = len(df[df.genre == genre]) - 1

    # Skip if recall is undefined
    if total == 0:
        return None

    _, idx = index.search(emb[i].reshape(1,-1), k+1)
    correct = np.sum(df.iloc[idx[0][1:]].genre == genre)
    return correct / total



In [38]:
sample = np.random.choice(len(df), 300)

precisions = []
recalls = []

for i in sample:
    precisions.append(precision_at_k(i))
    
    r = recall_at_k(i)
    if r is not None:
        recalls.append(r)

print("Precision@10:", np.mean(precisions))
print("Recall@10:", np.mean(recalls))
print("Valid recall samples:", len(recalls), "/", len(sample))


Precision@10: 0.04555555555555556
Recall@10: 0.025116827485378895
Valid recall samples: 281 / 300


In [33]:
import time
from sklearn.metrics.pairwise import cosine_similarity

def brute_force_recommend(idx, k=10):
    query = emb[idx].reshape(1, -1)
    sims = cosine_similarity(query, emb)[0]
    top_idx = sims.argsort()[-k-1:-1][::-1]
    return top_idx

# Measure time
start = time.time()
for i in range(50):   # 50 queries
    brute_force_recommend(i)
end = time.time()

print("Brute-force total time:", end - start)
print("Avg time per query:", (end - start)/50)


Brute-force total time: 1.9473824501037598
Avg time per query: 0.038947649002075195


In [34]:
def faiss_recommend(idx, k=10):
    query = emb[idx].reshape(1, -1).astype("float32")
    _, indices = index.search(query, k+1)
    return indices[0][1:]

start = time.time()
for i in range(50):
    faiss_recommend(i)
end = time.time()

print("FAISS total time:", end - start)
print("Avg time per query:", (end - start)/50)


FAISS total time: 0.0010008811950683594
Avg time per query: 2.0017623901367186e-05
